# Task 2 — Biggest Gains & Declines in Happiness (Slopegraphs, Altair)

Builds the two Task 2 slopegraphs for the **Drivers of Happiness** site:

1. **Biggest gains** — the 5 countries whose life evaluation rose the most over ~20 years
2. **Biggest declines** — the 5 countries whose life evaluation fell the most

**Method note:** single WHR years are noisy, so instead of comparing 2006 to 2025 directly we compare **3-year endpoint windows** (average of 2006–08 vs. average of 2023–25) and require at least 2 observations in each window. This keeps one unusual survey year from crowning a winner.

Both charts are exported as standalone HTML at the end, ready to drop into `charts/` and iframe into `index.html` side by side in the site's `viz-row`. They use the same site theme as Task 1, with the green/red poles of the map's color scale encoding the direction of change (the pair passes colorblind-separation and contrast checks; country names are also written directly on every line, so color is never the only cue).

**Running in Google Colab:** upload this notebook, run all cells — the data-loading cell will prompt you to upload `whr_2006_2025_cleaned.csv` (it lives in `charts/data/` in the repo). Running locally from the repo root works too and skips the upload.

In [ ]:
%pip install -q "altair>=5.0"

## Load the data

Reads from the repo if the notebook is run locally; otherwise prompts for an upload in Colab.

In [ ]:
import os
import pandas as pd

CSV_PATH = "charts/data/whr_2006_2025_cleaned.csv"  # path when run from the repo root

if os.path.exists(CSV_PATH):
    df_raw = pd.read_csv(CSV_PATH)
elif os.path.exists("whr_2006_2025_cleaned.csv"):
    df_raw = pd.read_csv("whr_2006_2025_cleaned.csv")
else:
    from google.colab import files  # only exists in Colab
    print("Upload whr_2006_2025_cleaned.csv")
    uploaded = files.upload()
    df_raw = pd.read_csv(next(iter(uploaded)))

print(df_raw.shape)
df_raw[["year", "country", "region", "life_evaluation"]].head()

## Find the biggest movers

For every country with enough data in both endpoint windows, compute the change in average life evaluation, then take the top and bottom 5.

In [ ]:
START_LABEL, END_LABEL = "2006–08", "2023–25"

d = df_raw.dropna(subset=["life_evaluation"])
start = d[d.year.between(2006, 2008)].groupby("country").life_evaluation.agg(["mean", "count"])
end = d[d.year.between(2023, 2025)].groupby("country").life_evaluation.agg(["mean", "count"])

m = start.join(end, lsuffix="_start", rsuffix="_end", how="inner")
m = m[(m.count_start >= 2) & (m.count_end >= 2)]  # ≥2 observations per window
m["change"] = m.mean_end - m.mean_start
m = m.join(d.groupby("country").region.first())

gains = m.nlargest(5, "change")
declines = m.nsmallest(5, "change")

print(f"{len(m)} countries have data in both windows")
pd.concat([gains, declines])[["mean_start", "mean_end", "change"]].round(2)

## Shape the data for the slopegraphs

Each country becomes two rows (one per endpoint). Endpoint labels get a small greedy **dodge** so near-tied values don't overlap — e.g. Latvia, Nicaragua, and the Philippines all end within 0.16 points of each other.

In [ ]:
Y_DOMAIN = [3, 7]   # shared across both panels so the slopes are comparable
LABEL_GAP = 0.17    # min vertical gap between endpoint labels, in score units

def dodge(values, gap=LABEL_GAP, lo=Y_DOMAIN[0], hi=Y_DOMAIN[1]):
    """Nudge label y-positions apart so near-tied endpoints stay readable."""
    order = sorted(range(len(values)), key=lambda i: -values[i])
    out = dict.fromkeys(order)
    prev = hi + gap
    for i in order:
        y = min(values[i], prev - gap)
        out[i] = y
        prev = y
    low = min(out.values())
    if low < lo:  # ran off the bottom — shift the whole block back up
        out = {i: y + (lo - low) for i, y in out.items()}
    return [out[i] for i in range(len(values))]

def slope_data(m):
    m = m.reset_index()
    m["label_start_y"] = dodge(m.mean_start.tolist())
    m["label_end_y"] = dodge(m.mean_end.tolist())
    rows = []
    for _, r in m.iterrows():
        common = dict(country=r.country, region=r.region, change=r.change,
                      start_score=r.mean_start, end_score=r.mean_end)
        rows.append(dict(common, period=START_LABEL, score=r.mean_start, label_y=r.label_start_y))
        rows.append(dict(common, period=END_LABEL, score=r.mean_end, label_y=r.label_end_y))
    return pd.DataFrame(rows)

gains_data, declines_data = slope_data(gains), slope_data(declines)
gains_data.head(4)

## Build the slopegraphs

Design choices:
- **One hue per panel** — green for gains, red for declines, the poles of the Task 1 map scale, so direction reads instantly and the two tasks feel like one system. Identity is carried by **direct labels** on every line (country + end score on the right, start score on the left), never by color alone.
- **Shared y-domain `[3, 7]`** across the two panels, so a steep line in one panel means the same thing in the other.
- **2px lines, white-ringed endpoint dots**, recessive grid, and the site's font/ink colors via the same `site_theme()` as Task 1.
- **Tooltips** on lines and points carry country, region, both window scores, and the signed change.

In [ ]:
import altair as alt

SITE_FONT = '-apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif'
INK, INK_SOFT, LINE = "#2f3542", "#6b7280", "#ece4d4"
GREEN, RED = "#1a9850", "#d73027"

def site_theme(chart):
    """Match the chart chrome to the website's styles.css (same as Task 1)."""
    return (
        chart.configure(font=SITE_FONT, background="white")
        .configure_view(stroke=None)
        .configure_title(
            anchor="start", color=INK, fontSize=17, fontWeight=700,
            subtitleColor=INK_SOFT, subtitleFontSize=12, subtitlePadding=6, offset=14,
        )
        .configure_axis(
            labelColor=INK_SOFT, titleColor=INK, labelFontSize=11, titleFontSize=12,
            gridColor=LINE, domainColor=LINE, tickColor=LINE,
        )
    )

def slopegraph(data, color, title, subtitle):
    x = alt.X(
        "period:N", title=None, sort=[START_LABEL, END_LABEL],
        scale=alt.Scale(padding=0.35),
        axis=alt.Axis(labelAngle=0, labelFontSize=12, labelColor=INK, domain=False, ticks=False),
    )
    y = alt.Y("score:Q", title="Life evaluation (0–10)",
              scale=alt.Scale(domain=Y_DOMAIN), axis=alt.Axis(tickCount=5))
    tooltip = [
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("region:N", title="Region"),
        alt.Tooltip("start_score:Q", title=START_LABEL, format=".2f"),
        alt.Tooltip("end_score:Q", title=END_LABEL, format=".2f"),
        alt.Tooltip("change:Q", title="Change", format="+.2f"),
    ]

    base = alt.Chart(data)
    lines = base.mark_line(strokeWidth=2, color=color).encode(
        x=x, y=y, detail="country:N", tooltip=tooltip)
    points = base.mark_point(filled=True, size=70, color=color, stroke="white", strokeWidth=1.5).encode(
        x=x, y=y, detail="country:N", tooltip=tooltip)
    left_vals = base.transform_filter(alt.datum.period == START_LABEL).mark_text(
        align="right", dx=-10, fontSize=11, color=INK_SOFT
    ).encode(x=x, y=alt.Y("label_y:Q", scale=alt.Scale(domain=Y_DOMAIN)),
             text=alt.Text("score:Q", format=".1f"))
    right_names = base.transform_filter(alt.datum.period == END_LABEL).transform_calculate(
        label='datum.country + "  " + format(datum.score, ".1f")'
    ).mark_text(align="left", dx=10, fontSize=11, fontWeight=600, color=INK).encode(
        x=x, y=alt.Y("label_y:Q", scale=alt.Scale(domain=Y_DOMAIN)), text="label:N")

    return (lines + points + left_vals + right_names).properties(
        width=270, height=360,
        padding={"left": 5, "right": 15, "top": 5, "bottom": 5},
        title=alt.Title(title, subtitle=subtitle),
    )

SUBTITLE = "Average life evaluation, 3-year windows. Hover a line for details."

gains_chart = site_theme(slopegraph(gains_data, GREEN, "Biggest gains in happiness", SUBTITLE))
gains_chart

In [ ]:
declines_chart = site_theme(slopegraph(declines_data, RED, "Biggest declines in happiness", SUBTITLE))
declines_chart

## Export standalone HTML for the website

Plain `chart.save()` — no custom template needed here (no play button). Both files are small (~8 KB) because only 10 countries' worth of data is embedded. In Colab this cell also triggers browser downloads.

In [ ]:
gains_chart.save("task2_slopegraph_gains.html", embed_options={"actions": False})
declines_chart.save("task2_slopegraph_declines.html", embed_options={"actions": False})
print("Saved task2_slopegraph_gains.html and task2_slopegraph_declines.html")

try:
    from google.colab import files
    files.download("task2_slopegraph_gains.html")
    files.download("task2_slopegraph_declines.html")
except ImportError:
    pass  # running locally — files are next to the notebook

## Embedding into the site

Move the two exported files into `charts/`, then replace the two slopegraph placeholders inside Task 2's `viz-row` in `index.html`:

```html
<div class="viz-row">
  <iframe src="charts/task2_slopegraph_gains.html" class="viz-frame"
          loading="lazy" style="min-height: 500px"
          title="Five countries with the biggest gains in happiness"></iframe>
  <iframe src="charts/task2_slopegraph_declines.html" class="viz-frame"
          loading="lazy" style="min-height: 500px"
          title="Five countries with the biggest declines in happiness"></iframe>
</div>
```

Still to come for Task 2 (left as placeholders on the site): the country selector and the per-country explanatory-factors table.